# Phase 0 Demo 01: Fetch, Store, Validate

This notebook verifies the Phase 0 data path on a small OKX public-data sample:
fetch OHLCV and funding, validate OHLCV, save parquet shards, then load them back.
Demo data is written under `data/cache/phase0_demo`, which is ignored by git.

In [1]:
from pathlib import Path

import pandas as pd

from data.fetcher import fetch_funding_rate_history, fetch_ohlcv
from data.storage import load_funding, load_ohlcv, save_funding, save_ohlcv
from data.validate import validate_ohlcv

In [2]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

symbol = "BTC/USDT:USDT"
until = pd.Timestamp.now("UTC").normalize()
since = until - pd.Timedelta(days=14)
data_dir = project_root / "data" / "cache" / "phase0_demo"
data_dir.mkdir(parents=True, exist_ok=True)

ohlcv = fetch_ohlcv(symbol, since=since, until=until)
funding = fetch_funding_rate_history(symbol, since=since, until=until)

report = validate_ohlcv(ohlcv, symbol=symbol)
assert report.ok, report.summary()

save_ohlcv(ohlcv, symbol, data_dir=data_dir)
save_funding(funding, symbol, data_dir=data_dir)

loaded_ohlcv = load_ohlcv(symbol, data_dir=data_dir)
loaded_funding = load_funding(symbol, data_dir=data_dir)
assert len(loaded_ohlcv) == len(ohlcv)
assert len(loaded_funding) == len(funding)

2026-05-30 23:51:47.802 | INFO     | data.fetcher:fetch_ohlcv:94 - fetch_ohlcv BTC/USDT:USDT 1d: 15 根 [2026-05-17 ~ 2026-05-31]


2026-05-30 23:51:49.896 | INFO     | data.fetcher:fetch_funding_rate_history:147 - fetch_funding BTC/USDT:USDT: 43 条 [2026-05-17 ~ 2026-05-31]


2026-05-30 23:51:49.932 | INFO     | data.storage:_save_sharded:66 - save BTC/USDT:USDT 1d: 15 根 -> 1 个分片


2026-05-30 23:51:49.938 | INFO     | data.storage:_save_sharded:66 - save BTC/USDT:USDT funding: 43 根 -> 1 个分片


In [3]:
pd.DataFrame(
    [
        {
            "symbol": symbol,
            "ohlcv_rows": len(ohlcv),
            "funding_rows": len(funding),
            "ohlcv_start": ohlcv.index.min(),
            "ohlcv_end": ohlcv.index.max(),
            "funding_start": loaded_funding.index.min(),
            "funding_end": loaded_funding.index.max(),
            "validation_ok": report.ok,
            "data_dir": str(data_dir.relative_to(project_root)),
        }
    ]
)

,symbol,ohlcv_rows,funding_rows,ohlcv_start,ohlcv_end,funding_start,funding_end,validation_ok,data_dir
0,BTC/USDT:USDT,15,43,2026-05-17 00:00:00+00:00,2026-05-31 00:00:00+00:00,2026-05-17 00:00:00+00:00,2026-05-31 00:00:00+00:00,True,data/cache/phase0_demo
